In [1]:
import pandas as pd

from control_totals.util import Pipeline

p = Pipeline('configs')

In [247]:
with pd.HDFStore('data/pipeline.h5') as store:
    tables = store.keys()
tables

['/adjusted_emp_change_targets_calculations',
 '/adjusted_emp_change_targets_no_res_con',
 '/adjusted_emp_change_targets_no_res_con_calculations',
 '/adjusted_emp_change_targets_res_con',
 '/adjusted_emp_change_targets_res_con_calculations',
 '/adjusted_emp_targets_no_res_con',
 '/adjusted_king_targets',
 '/adjusted_total_pop_change_targets',
 '/adjusted_units_change_targets',
 '/block_2010_control_area_xwalk',
 '/block_control_area_xwalk',
 '/blocks',
 '/blocks_2010',
 '/control_areas',
 '/control_target_xwalk',
 '/control_totals',
 '/dec_block_data',
 '/decennial_by_control_area',
 '/employment_2018_by_control_area',
 '/employment_2019_by_control_area',
 '/employment_2020_by_control_area',
 '/extrapolated_targets',
 '/king_targets',
 '/kitsap_targets',
 '/ofm_block_2019',
 '/ofm_block_2019_by_control_area',
 '/ofm_parcel_control_area_xwalk',
 '/ofm_parcelized_2018',
 '/ofm_parcelized_2018_by_control_area',
 '/ofm_parcelized_2019',
 '/ofm_parcelized_2019_by_control_area',
 '/ofm_parce

In [16]:
test = p.get_table('adjusted_emp_change_targets_calculations')
test.loc[test.target_id == 176]

,target_id,start,emp_chg,emp_chg_adj,county_id,control_id_2020,control_name_2020,Emp_TotNoMil_2020,Emp_ConRes_2020,TotEmpNoMil-ResCon_2020,Emp_Military_2020,Emp_Total_2020,emp_2044
137,176,2019,4427.0,4896.754601,53061,690,Snohomish RuralTulalip TribeSnohomish Rural Ag...,26050,7890,18160,0,26050,30947


In [2]:
from control_totals.steps.legacy.emp_chg_targets import load_targets

In [3]:
df = load_targets(p)

In [5]:
df.loc[df.target_id == 1]

,target_id,start,emp_chg,emp_chg_adj,county_id,control_id_2020,control_name_2020,Emp_TotNoMil_2020,Emp_ConRes_2020,TotEmpNoMil-ResCon_2020,Emp_Military_2020,Emp_Total_2020
37,1,2019,70000.0,75395.320736,53033,1,Bellevue,155023,8469,146554,0,155023


In [7]:
from control_totals.steps.emp_chg_targets_no_res_con import load_targets

df = load_targets(p)

base_year = p.settings['base_year']
end_year = p.settings['targets_end_year']

emp_end_year_col = f'emp_{end_year}'
emp_no_mil_base_year_col = f'Emp_TotNoMil_{base_year}'



In [11]:
df[emp_end_year_col] = (df[emp_no_mil_base_year_col] + df['emp_chg_adj'])

In [12]:
df.loc[df[emp_end_year_col].isna()]

,target_id,start,emp_chg,emp_chg_adj,county_id,control_id_2020,control_name_2020,Emp_TotNoMil_2020,Emp_ConRes_2020,TotEmpNoMil-ResCon_2020,Emp_Military_2020,Emp_Total_2020,emp_2044
46,49,2019,0.0,NaN,53033,49,Black Diamond PAA,0,0,0,0,0,NaN
50,57,2019,0.0,NaN,53033,57,Newcastle PAA,0,0,0,0,0,NaN
85,157,2019,76.0,NaN,53061,157,Darrington UGA,0,0,0,0,0,NaN
87,159,2019,3.0,NaN,53061,159,Granite Falls UGA,0,0,0,0,0,NaN
91,167,2019,0.0,NaN,53061,167,Mountlake Terrace MUGA,0,0,0,0,0,NaN
94,174,2019,1.0,NaN,53061,174,Sultan UGA,0,0,0,0,0,NaN
95,175,2019,32.0,NaN,53061,175,Woodway MUGA,0,0,0,0,0,NaN


In [ ]:

# multiply the base year resource and construction employment percentage by the employment change target
df['res_con_target_pct'] = df['res_con_pct'] * df['emp_chg']

# sum targets to county level
df['emp_chg_cnty_sum'] = df.groupby('county_id')['emp_chg'].transform('sum')
# get resource and construction employment growth percentage from ref projection
res_con_emp_growth_pct = p.settings['res_con_emp_growth_pct']
# calculate resource and construction employment change target for county
df['res_con_emp_chg_cnty_sum'] = df['emp_chg_cnty_sum'] * res_con_emp_growth_pct
# allocate county resource and construction employment change target to targets based on their resource and construction target percentage
df['res_con_emp_chg_target'] = df['res_con_target_pct'] * df['res_con_emp_chg_cnty_sum'] / df.groupby('county_id')['res_con_target_pct'].transform('sum')

# adjust employment change target by adding resource and construction employment change target
df['emp_chg_adj_res_con'] = df['emp_chg_adj'] + df['res_con_emp_chg_target']

# add with base year employment to get adjusted employment total
emp_no_mil_end_year_col = f'emp_{end_year}'
df[emp_no_mil_end_year_col] = (df[emp_no_mil_base_year_col] + df['emp_chg_adj_res_con']).fillna(0).round(0).astype(int)


Summing employment estimates for year 2019 to target areas...
Summing employment estimates for year 2020 to target areas...


,target_id,emp_chg,start,emp_2019,emp_2020
0,1,70000.0,2019,150598,146554
1,2,169500.0,2019,642647,658939
2,3,19520.0,2019,44204,43366
3,4,9500.0,2019,16928,16829
4,5,4770.0,2019,12837,12682
...,...,...,...,...,...
96,162,707.0,2019,905,898
97,169,1434.0,2019,1895,2174
98,163,1006.0,2019,3610,3576
99,168,1584.0,2019,6387,6667


In [199]:
base_year = p.settings['base_year']

# combine county targets
df = combine_targets(pipeline, 'emp', emp_target_type='no_res_con')

# get unique start years in the targets
start_years = df['start'].unique().tolist()

# get estimates for all start years and base year amd merge to targets
est_all_years = get_estimates_all_years(p, start_years, 'emp', 'employment')
if not est_all_years.empty:
    df = df.merge(est_all_years, on=['target_id'], how='left')

# load resource and construction employment targets from settings
res_con_targets = p.settings['resource_construction_emp_targets']

# change 2019 to 2020 emp w/ res con, no military
df['emp_chg_no_military'] = df['Emp_TotNoMil_2020'] - df['Emp_TotNoMil_2019']
# percent of employment 2020 that is res con
df['pct_res_con'] = df['Emp_ConRes_2020'] / df['Emp_TotNoMil_2020']
# res con job change based on emp target
df['res_con_chg'] = df['emp_chg'] * df['pct_res_con']
# normalize resource and construction employment changes by county using the county targets from settings
df['res_con_chg_norm'] = df['county_id'].map(res_con_targets) * df['res_con_chg'] / df.groupby('county_id')['res_con_chg'].transform('sum')
# if the adjusted employment change is negative, fall back to the original employment change
# otherwise use the adjusted employment change
emp_chg_adj = df['emp_chg'] - df['emp_chg_no_military'] + df['res_con_chg_norm']
df['emp_chg_adj'] = np.where(emp_chg_adj < 0, df['emp_chg'], emp_chg_adj)
# return adjusted targets
return df[['target_id','start','emp_chg','emp_chg_adj']]

Summing employment estimates for year 2019 to target areas...
Summing employment estimates for year 2020 to target areas...


In [ ]:
# change 2019 to 2020 emp w/ res con, no military
df['emp_chg_no_military'] = df['Emp_TotNoMil_2020'] - df['Emp_TotNoMil_2019']
# percent of employment 2020 that is res con
df['pct_res_con'] = df['Emp_ConRes_2020'] / df['Emp_TotNoMil_2020']
# res con job change based on emp target
df['res_con_chg'] = df['emp_chg'] * df['pct_res_con']
# normalize resource and construction employment changes by county using the county targets from settings
df['res_con_chg_norm'] = df['county_id'].map(res_con_targets) * df['res_con_chg'] / df.groupby('county_id')['res_con_chg'].transform('sum')
# if the adjusted employment change is negative, fall back to the original employment change
# otherwise use the adjusted employment change
emp_chg_adj = df['emp_chg'] - df['emp_chg_no_military'] + df['res_con_chg_norm']
df['emp_chg_adj'] = np.where(emp_chg_adj < 0, df['emp_chg'], emp_chg_adj)
# return adjusted targets
df[['target_id','start','emp_chg','emp_chg_adj']]

In [ ]:
# percent of employment 2020 that is res con
df['pct_res_con'] = df['Emp_ConRes_2020'] / df['Emp_TotNoMil_2020']
# res con job change based on emp target
df['res_con_chg'] = df['emp_chg'] * df['pct_res_con']
# normalize resource and construction employment changes by county using the county targets from settings
df['res_con_chg_norm'] = df['county_id'].map(res_con_targets) * df['res_con_chg'] / df.groupby('county_id')['res_con_chg'].transform('sum')
# if the adjusted employment change is negative, fall back to the original employment change
# otherwise use the adjusted employment change
emp_chg_adj = df['emp_chg'] - df['emp_chg_no_military'] + df['res_con_chg_norm']
df['emp_chg_adj'] = np.where(emp_chg_adj < 0, df['emp_chg'], emp_chg_adj)
# return adjusted targets
df[['target_id','start','emp_chg','emp_chg_adj']]

,target_id,start,emp_chg,emp_chg_adj
0,1,2019,70000.0,75395.320736
1,2,2019,169500.0,156244.955044
2,3,2019,19520.0,20517.160124
3,4,2019,9500.0,9827.836990
4,5,2019,4770.0,5041.847552
...,...,...,...,...
96,162,2019,707.0,744.081749
97,169,2019,1434.0,1135.052739
98,163,2019,1006.0,1195.956189
99,168,2019,1584.0,1304.411873
